# Teas as ingredient vectors

**The question this notebook exists to answer.** Herbatka has 24 approved teas and 64
ingredients, and `tea_ingredient` already records which goes in which — with a percentage
and a primary flag. That is a matrix. If two teas sit close together in it, the app can say
*"teas like this one"* on day one, with no reviews, no users and nothing to train.

The question is not *can* we compute a similarity — of course we can, it is one line. It is
**which weighting produces neighbour lists a tea drinker would agree with**, and how we
would know if it did not.

Two things make this a good first piece of data science for this project:

- The data is **real**. Somebody hand-wrote those 23 recipes. Contrast the review table:
  14 rows, all from `seed/people.py`, i.e. fiction. A model fitted to it would be measuring
  its author's imagination.
- The output is **cold-start proof**. Content similarity needs no behaviour, so it works
  the day the app launches — which is exactly when a recommender that needs behaviour does not.

> **Status: a plan.** Sections 1 and 7 run. Sections 2–6 are stubs with hints, to be filled
> in by hand. **Run section 1 first and read the result cell under it** — on today's
> catalogue it already changes what is worth building. That is deliberate — the whole value here is in the four decisions those
> sections contain, and importing `TfidfVectorizer` would make all four of them silently
> for you.

## 0. Setup

`uv sync` in `analysis/`, then pick the `analysis/.venv` kernel — this repository now has
**three** virtualenvs (`api/`, `vision/`, `analysis/`) and no editor guesses correctly
between them. If the next cell raises `ModuleNotFoundError: herbatka_analysis`, that is
which mistake it is.

The database must be up: `make db` from the repository root, and `make seed` if you have
not already.

In [ ]:
import numpy as np
import pandas as pd

from herbatka_analysis import db, paths

paths.ensure()
pd.set_option("display.max_columns", 30)

# Read-only by construction — see the docstring in db.engine().
frames = db.catalogue()
teas, ingredients, links = frames["teas"], frames["ingredients"], frames["links"]

print(f"{len(teas):>4} teas")
print(f"{len(ingredients):>4} ingredients")
print(f"{len(links):>4} tea-ingredient links")
print(f"     density: {len(links) / (len(teas) * len(ingredients)):.1%} of the matrix is non-zero")

## 1. Look at it before you model it

Skipping this step is the most common way to waste a week. Three things are worth knowing
before any similarity is computed, because each one changes what a sensible weighting looks
like:

1. **How sparse is it?** At ~3% density, most tea pairs share *no* ingredient at all, so
   their similarity is exactly zero and the ranking among them is arbitrary. Knowing how
   many pairs are in that state tells you whether "similar teas" can be a headline feature
   or only a sometimes-feature.
2. **How often does each ingredient appear?** An ingredient in one tea and an ingredient in
   twelve carry wildly different amounts of information. This is the whole of section 3.
3. **Is `percentage` actually populated?** It is nullable. If it is null for most rows, a
   percentage-weighted model is not an option — it is a fantasy about data you do not have.

Run the cell, then read the three numbers as *constraints*, not as trivia.

In [ ]:
# 1a. How many ingredients does a tea have, and how many teas does an ingredient appear in?
per_tea = links.groupby("tea_id").size()
per_ingredient = links.groupby("ingredient_id").size()

print("ingredients per tea:", per_tea.describe()[["min", "50%", "max"]].to_dict())
print("teas per ingredient:", per_ingredient.describe()[["min", "50%", "max"]].to_dict())

# 1b. The commonest ingredients — these are the ones that will dominate an unweighted model.
common = (
    links.merge(ingredients, left_on="ingredient_id", right_on="id")
    .groupby("name")
    .size()
    .sort_values(ascending=False)
)
print("\ntop 8 by appearances:\n", common.head(8).to_string())

# 1c. Is percentage usable at all?
print(f"\npercentage populated: {links['percentage'].notna().mean():.0%} of links")

### What section 1 says today — read this before writing section 2

Measured against the seeded catalogue: **24 teas, 64 ingredients, 51 links.**

| | |
|---|---|
| ingredients per tea | min 1, **median 2**, max 6 |
| matrix density | 3.3% |
| **tea pairs sharing ≥ 1 ingredient** | **25 of 276 — 9.1%** |
| ingredients used in exactly one tea | 25 of 64 |
| `percentage` populated | 100% ✅ |

The third row is the one to sit with. **For 91% of tea pairs the cosine similarity is
exactly zero** — not low, zero — so the ordering among them is not weak, it is arbitrary.
Averaged out, each tea has about two other teas it can be said to resemble at all.

The fourth row makes the section 3 trap concrete before you meet it: 25 ingredients appear
in exactly one tea, so under unsmoothed IDF they carry the maximum available weight, and
the model turns into a novelty detector.

The fifth row is the good news, and it is worth noting how rarely it goes this way: every
link has a percentage, so percentage weighting is genuinely on the table rather than being
a fantasy about data you do not have.

**What this means — and what it does not.** It does not mean the approach is wrong. It
means the bottleneck is not the model. A median of two ingredients per tea is a *stub* of a
recipe, not a recipe: a real Christmas blend has eight, and the matrix cannot show a
resemblance that the catalogue never recorded. So the highest-leverage work right now is in
`api/app/seed/data.py`, writing fuller recipes — and no amount of cleverness in sections 3–4
substitutes for it.

Re-run section 1 after enriching the catalogue. Once the "pairs sharing ≥ 1 ingredient"
figure clears roughly 50%, sections 2–6 are worth finishing; much below that they are
arithmetic over a matrix of zeros.

This is section 6's stopping rule arriving early — which is the *good* case, because it
arrived before a fortnight went into weighting schemes rather than after.

## 2. Build the matrix

One row per tea, one column per ingredient. The only real decision is what goes in a cell,
and section 3 is about that — so build this one to take the cell value as an argument rather
than hard-coding it, and you get to compare weightings for free instead of rewriting this
function three times.

**Watch the row and column order.** A matrix whose rows are in `groupby` order and whose
labels are in `teas.slug` order is the bug that produces *plausible* nonsense: every
neighbour list is well-formed, and every one of them is wrong. Build the index explicitly
from the same frame you label with, and assert the shape.

In [ ]:
def build_matrix(teas: pd.DataFrame, ingredients: pd.DataFrame, links: pd.DataFrame,
                 value: str = "binary") -> tuple[np.ndarray, list[str], list[str]]:
    """Return (matrix, tea_slugs, ingredient_slugs) with matrix.shape == (teas, ingredients).

    `value` is one of:
      "binary"     — 1.0 if the ingredient is present
      "percentage" — the recorded percentage, with a sensible fallback where it is null
      "primary"    — a larger weight for is_primary rows

    Hints:
      - Map id -> row index with a dict built from `teas["id"]`, same for ingredients.
      - `np.zeros((len(teas), len(ingredients)))`, then one pass over `links.itertuples()`.
      - For "percentage", decide *and write down* what a null means. Dropping the row and
        treating it as an equal share of the unaccounted remainder are both defensible;
        silently using 0.0 is not, because it says "contains none of this".
    """
    raise NotImplementedError


# X, tea_slugs, ing_slugs = build_matrix(teas, ingredients, links, value="binary")
# assert X.shape == (len(teas), len(ingredients))

## 3. The decision that matters: weighting

Here is the problem, in your own data. *Green tea leaf* appears in five of the 24 teas and
*black tea leaf* in four. Under a binary matrix, every pair of those five gets credit for
sharing it — so a matcha and a jasmine green look similar because they are both, well, tea.
Meanwhile 25 ingredients appear in exactly one tea each and contribute to no pair at all.

That is backwards. An ingredient is informative in proportion to its **rarity**, which is
what inverse document frequency says, and TF-IDF is just that idea written down:

$$\text{idf}(i) = \ln\frac{N}{1 + n_i} \qquad w_{t,i} = \text{tf}_{t,i} \cdot \text{idf}(i)$$

with $N$ = number of teas and $n_i$ = number of teas containing ingredient $i$.

**The trap, specific to a catalogue this small.** With $N = 24$ and 25 ingredients appearing
exactly once, an ingredient used a single time gets the largest IDF available — so a tea
containing one exotic ingredient becomes *maximally distinctive*, and its neighbour list
collapses to "other teas with that one weird thing". IDF is genuinely noisy at this sample
size, and here it is noisy for a third of the vocabulary. Smoothing (the $1 + n_i$, or capping the weight) is not decoration here —
it is what keeps the model from being a novelty detector. Try it both ways and look.

In [ ]:
def idf(X_binary: np.ndarray, smooth: bool = True) -> np.ndarray:
    """Inverse document frequency per ingredient. Shape (n_ingredients,).

    Hints:
      - Document frequency is `X_binary.sum(axis=0)` — count teas, not grams.
      - np.log(N / (1 + df)) with smooth=True; np.log(N / df) without, and mind the
        divide-by-zero for ingredients no tea uses (65 ingredients, ~23 teas: there will
        be some).
      - Sanity check before you go further: the highest-IDF ingredient should be something
        you recognise as unusual, and the lowest should be black tea leaf or similar. If it
        is the other way round, the axis is wrong.
    """
    raise NotImplementedError


def weighted(X: np.ndarray, weights: np.ndarray) -> np.ndarray:
    """Scale every column of X by its weight. One line; the point is to name the operation."""
    raise NotImplementedError

## 4. Similarity

Cosine, not Euclidean — and the reason is a fact about tea, not a convention. A single-origin
Darjeeling has one ingredient; a Christmas blend has twelve. Euclidean distance makes those
two far apart *because of the count*, which is not what anybody means by "not similar".
Cosine divides it out by normalising every row to unit length, so it compares **composition**
rather than length of the ingredient list.

Once rows are L2-normalised, the entire similarity matrix is `Xn @ Xn.T`. Two lines.

Then do the part that is not code: **read all 23 neighbour lists**. You know about tea; the
matrix does not. At this size, your own eye is a better instrument than any metric, and it is
the only step that catches a whole class of error a number will happily average away.

In [ ]:
def l2_normalise(X: np.ndarray) -> np.ndarray:
    """Scale each row to unit length. Guard the all-zero rows — a tea with no ingredient
    links divides by zero and quietly seeds NaNs through the whole similarity matrix."""
    raise NotImplementedError


def neighbours(S: np.ndarray, tea_slugs: list[str], slug: str, k: int = 5) -> pd.DataFrame:
    """The k most similar teas to `slug`, excluding itself.

    Hints:
      - `np.fill_diagonal(S, -np.inf)` on a *copy*, or every tea's best friend is itself.
      - `np.argsort(-row)[:k]`.
      - Return the score alongside the name: a neighbour at 0.98 and one at 0.04 mean very
        different things, and a bare list hides that.
    """
    raise NotImplementedError

## 5. How would you know if it were bad?

The uncomfortable part, and the reason most side-project recommenders are never actually
evaluated: **there are no labels.** Nobody has written down which teas are similar, so there
is no accuracy to compute. Three ways out, in increasing order of honesty:

**(a) Eyeball the 23 lists.** Necessary, and genuinely informative at this size. But it is
not a number, it cannot detect a regression, and you will unconsciously grade your own
homework.

**(b) Agreement with `tea_type`, against chance.** You have a label you did not create for
this purpose: `tea.tea_type` (green/black/oolong/…). A good ingredient-space model should
put a green tea's nearest neighbour in *green* more often than picking at random would. This
is computable today, it is a single number, and the baseline is not a guess — it is the
type distribution in the catalogue.

Note carefully what this measures: it is a **sanity check, not the objective.** A model
scoring 100% here has merely rediscovered `tea_type`, which you already had in a column —
and a rooibos-and-vanilla sitting next to a black-and-vanilla may be a genuinely good
recommendation that this metric marks wrong. Use it to catch a *broken* model, not to pick
the best one.

**(c) Held-out ingredient prediction.** The one I would actually build. Hide one ingredient
from one tea, then ask: do that tea's nearest neighbours contain the hidden ingredient more
often than the catalogue's base rate? This needs no human labels, has a principled baseline,
runs over every (tea, ingredient) pair for ~100 evaluations from 23 teas, and measures the
thing you actually want — *does proximity in this space predict composition you did not
show it*. It is leave-one-out cross-validation, applied to a matrix instead of a model.

In [ ]:
def type_agreement(S: np.ndarray, teas: pd.DataFrame, tea_slugs: list[str]) -> dict[str, float]:
    """Fraction of teas whose nearest neighbour shares their tea_type, plus the chance rate.

    Beware the zero rows: with 91% of pairs at similarity 0, "nearest neighbour" is a tie
    among most of the catalogue and argmax silently returns whichever index came first —
    which is alphabetical by slug, not similar. Count only teas with a non-zero best match,
    and report how many you had to skip.

    Hints:
      - Chance is not 1/n_types — the catalogue is unbalanced. For a tea of type k it is
        (count_k - 1) / (n_teas - 1); average that over teas for the baseline.
      - Return both numbers together. A bare 0.61 is unreadable without its baseline.
    """
    raise NotImplementedError


def holdout_ingredient_score(X_binary: np.ndarray, weight_fn, k: int = 5) -> dict[str, float]:
    """Leave-one-ingredient-out: can a tea's neighbours predict what you hid?

    Sketch:
      for each (tea, ingredient) link:
          copy X, zero that one cell
          recompute weights *from the modified matrix* — recomputing IDF is the point,
              because leaking the full-matrix IDF is exactly the subtle leak this design
              is meant to avoid
          find the tea's top-k neighbours
          score a hit if any neighbour contains the held-out ingredient
      return {"hit_rate": ..., "base_rate": ...}

    The baseline: how often would k teas picked at random contain that ingredient? For an
    ingredient in n_i of N teas, roughly 1 - C(N-1-n_i, k) / C(N-1, k). Beating it is the
    whole claim; not beating it means the space carries no information about composition.
    """
    raise NotImplementedError

## 6. The decision, written down before the result

Committing now is what stops next weekend from being a graph neural network you build because
you had already decided to build one.

- **Ship the simplest weighting that beats base rate on (c).** If binary already beats it,
  ship binary. "TF-IDF is more sophisticated" is not a reason; a higher hit rate is.
- **If no weighting beats base rate**, the ingredient space does not support similarity yet,
  and the honest conclusion is *not enough catalogue*. Given section 1 — 9% of pairs sharing
  anything, a median of two ingredients per tea — this is the likely outcome today, and the
  fix is fuller recipes rather than a better model. Ship nothing, enrich `seed/data.py`,
  re-run section 1, and only then come back here. A shipped recommender that is indistinguishable from random
  is worse than no recommender, because it teaches users to ignore the feature.
- **If percentage weighting wins but only 30% of links have a percentage**, do not ship it.
  A model that is good on the third of the catalogue with complete data and arbitrary on the
  rest is two features wearing one name.
- **Do not add embeddings, UMAP or a learned model in this notebook.** They are the answer to
  "cosine on IDF is not good enough", which is a sentence nobody can say yet.

## 7. Where this goes when it works

Two deliverables, both already fitting the shape of the repository:

### Store the edges, not the vectors

The instinct is to store the 64-dimensional vector per tea. Don't. Here is the schema, then
the reasoning:

```sql
CREATE TABLE tea_similarity (
    tea_id       UUID        NOT NULL REFERENCES tea(id) ON DELETE CASCADE,
    neighbour_id UUID        NOT NULL REFERENCES tea(id) ON DELETE CASCADE,
    rank         SMALLINT    NOT NULL,
    score        REAL        NOT NULL,
    method       TEXT        NOT NULL,   -- 'ingredient-idf-cosine-v1'
    computed_at  TIMESTAMPTZ NOT NULL,

    PRIMARY KEY (tea_id, neighbour_id),
    CONSTRAINT ck_tea_similarity_not_self  CHECK (tea_id <> neighbour_id),
    CONSTRAINT ck_tea_similarity_positive  CHECK (score > 0)
);

CREATE INDEX ix_tea_similarity_lookup ON tea_similarity (tea_id, rank);
```

**Why not the vectors.** A vector here is derived data with a *global* dependency: IDF is
computed across the whole catalogue, so approving one new tea changes all 24 vectors. A
`tea_vector` table would look incrementally updatable and would not be, and that gap between
appearance and truth is where a stale-data bug lives for a year. The edge table cannot mislead
you the same way, because the only honest way to maintain it is `DELETE` then re-`INSERT`
inside one transaction — which is exactly what a global recompute is.

**Why not `pgvector`.** It earns its keep with approximate nearest-neighbour search over
millions of rows. You have 24. The exact answer is a 24×24 matrix multiply that takes
microseconds, and an extension plus an index type plus a new query language to serve 120 rows
is theatre. Revisit somewhere north of 100k teas.

**`CHECK (score > 0)` is doing real work.** Section 1 found that 91% of tea pairs share no
ingredient at all. Under this constraint those pairs simply *cannot be stored* — so
`GET /teas/{slug}/similar` returns an empty list for a Sencha rather than five arbitrary teas
ranked by floating-point noise. The schema enforces the honest behaviour, instead of leaving
it to a `WHERE score > 0` that some future caller forgets. Make the UI's empty state good;
it will be the common case for a while.

**`method` and `computed_at` are not bookkeeping.** They are what lets you change the
weighting and still answer "why is this tea's neighbour list different from last week's" —
and what lets two methods coexist while you compare them.

**The write path.** The computation lives here, in numpy; the *table* is owned by an Alembic
migration in `api/`, so the schema keeps a single source of truth. `analysis/` is the only
writer, via a non-read-only engine used by that one job, and the API only ever reads. No numpy
in the API image, no model at request time — the same constraint `vision/README.md` states,
honoured the cheap way. `GET /teas/{slug}/similar` becomes one indexed select in
`app/services/catalog.py`.

**The one thing worth storing as coordinates** is the flavour map: two floats per tea, `map_x`
and `map_y`, from `np.linalg.svd`. Two columns on `tea`, not 64.

### The flavour map

Once teas are vectors, `np.linalg.svd` projects them to 2-D and the catalogue becomes a
picture — a genuinely lovely page for a tea app, and an honest one, because a projection makes
no claim beyond "these things sat near each other". Be warned that with 91% of pairs
orthogonal, the current map is mostly a spray of unrelated points; it gets a shape when the
blends outnumber the single-origins.

The cell below is the export, ready for when sections 2–6 pass their own test. It writes a
CSV rather than touching the database: the engine here is read-only on purpose, so
publishing is a separate, deliberate step in `api/`.

In [ ]:
def export_neighbours(S: np.ndarray, tea_slugs: list[str], k: int = 5) -> pd.DataFrame:
    """Long-format (tea_slug, rank, neighbour_slug, score) — one row per edge, ready to load.

    Long, not wide: a `neighbour_1 … neighbour_5` table cannot be joined, cannot be filtered
    by score, and needs a migration to become a top-10.
    """
    raise NotImplementedError


# out = export_neighbours(S, tea_slugs)
# out.to_csv(paths.OUTPUTS / "tea_similarity.csv", index=False)
# print(f"wrote {len(out)} edges")

## Next

1. Fill in sections 2–4 and read all 24 neighbour lists. Expect to find one obvious bug —
   most likely a row/column ordering slip, which looks like plausible nonsense rather than
   an error, or an argmax over a row of zeros returning tea number 0.
2. Implement (c) in section 5. It is the only cell that decides anything.
3. Apply the rule in section 6, whatever it says.
4. If it ships: the `tea_similarity` table and a `make similarity` target. If it does not:
   the catalogue needs to grow, and that is a useful finding rather than a failure.